In [1]:
import numpy as np
import pandas as pd
#from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, AutoModel
import re
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

2026-02-04 13:58:46.244208: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770213526.451683      26 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770213526.514710      26 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770213527.011617      26 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770213527.011662      26 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770213527.011665      26 computation_placer.cc:177] computation placer alr

In [2]:
# Test dataset
data_pd = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv').set_index('id')
data_pd

,text_id,line_start,line_end,transliteration
id,,,,
0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "/kaggle/input/byt5-first-10-epochs/transformers/default/6/byt5-akkadian/checkpoint-8800"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(DEVICE)

In [4]:
def preprocess_akkadian_text(text, is_translation=False):
    """
    Clean Akkadian transliteration or translation text for LLM training.
    
    Parameters:
    -----------
    text : str
        The raw Akkadian transliteration or translation text
    is_translation : bool
        If True, applies translation-specific cleaning
    
    Returns:
    --------
    str : Cleaned text ready for LLM training
    """
    
    # Handle special Unicode characters (convert to standard form)
    # Normalize special Akkadian characters
    char_replacements = {
        'á': 'a₂', 'à': 'a₃',
        'é': 'e₂', 'è': 'e₃',
        'í': 'i₂', 'ì': 'i₃',
        'ú': 'u₂', 'ù': 'u₃',
        'š': 'š', 'Š': 'Š',  # Keep š/Š as they are proper Unicode
        'Ṣ': 'Ṣ', 'ṣ': 'ṣ',  # Keep as proper Unicode
        'Ṭ': 'Ṭ', 'ṭ': 'ṭ',  # Keep as proper Unicode
        'Ḫ': 'H', 'ḫ': 'h',  # Convert Ḫ/ḫ to H/h as per instructions
    }
    
    for old, new in char_replacements.items():
        text = text.replace(old, new)
    
    # Handle subscript numbers - remove subscript formatting but keep numbers
    # Convert subscript numbers to regular numbers (₀-₉ → 0-9)
    subscript_to_normal = {
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
        'ₓ': 'x'  # Special subscript x
    }
    
    for sub, normal in subscript_to_normal.items():
        text = text.replace(sub, normal)
    
    # REMOVE modern scribal notations (transliteration specific)
    if not is_translation:
        # Remove: ! ? / : .
        text = re.sub(r'[!?/:.]', '', text)
        
        # Remove partially broken signs ˹ ˺
        text = text.replace('˹', '').replace('˺', '')
        
        # Remove content in square brackets but keep the text inside
        # [KÙ.BABBAR] → KÙ.BABBAR
        text = re.sub(r'\[([^\]]*)\]', r'\1', text)
        
        # Remove parentheses but keep content inside (for translations, we handle differently)
        text = re.sub(r'\(([^)]*)\)', r'\1', text)
        
        # Remove scribal insertions markers but keep the text
        text = re.sub(r'<([^>]*)>', r'\1', text)
        
        # Remove double pointy brackets (erroneous signs)
        text = re.sub(r'<<([^>]*)>>', r'\1', text)
    
    # REPLACE breaks and gaps
    # Single sign break
    text = re.sub(r'\[x\]|x{1,3}|\bxx\b', ' <gap> ', text)
    
    # Large/multiple breaks
    text = re.sub(r'\[\.\.\.\]|\.\.\.|\[…\]|…|\[\.\.\. \.\.\.\]', ' <big_gap> ', text)
    
    # Handle determinatives in curly brackets
    # Replace determinative markers with clean format
    determinatives = {
        r'\{d\}': 'DINGIR',  # god/deity
        r'\{mul\}': 'MUL',  # stars
        r'\{ki\}': 'KI',  # earth/place
        r'\{lu[₂2]?\}': 'LU',  # people/professions
        r'\{e[₂2]?\}': 'É',  # buildings
        r'\{uru\}': 'URU',  # settlements
        r'\{kur\}': 'KUR',  # lands/mountains
        r'\{mi\}': 'MUNUS',  # feminine
        r'\{m\}': 'M',  # masculine
        r'\{geš\}|\{ĝeš\}': 'GIŠ',  # wood/trees
        r'\{tug[₂2]?\}': 'TÚG',  # textiles
        r'\{dub\}': 'DUB',  # tablets/documents
        r'\{id[₂2]?\}': 'ÍD',  # canals/rivers
        r'\{mušen\}': 'MUŠEN',  # birds
        r'\{na[₄4]?\}': 'NA₄',  # stone
        r'\{kuš\}': 'KUŠ',  # skins/hides
        r'\{u[₂2]?\}': 'Ú',  # plants
    }
    
    for pattern, replacement in determinatives.items():
        text = re.sub(pattern, replacement, text)
    
    # Clean up determinatives that might appear with words
    # Example: a-lim{ki} → a-lim KI
    text = re.sub(r'(\w+)\{([^}]+)\}', r'\1 \2', text)
    
    # Handle line numbers and apostrophes
    # Remove line numbers like 1, 5, 10, 15' , 20'' etc.
    text = re.sub(r'\b\d+\'*\b', '', text)
    
    # Handle subscripted vowels in determinatives
    # Remove curly brackets from any remaining determinatives
    text = text.replace('{', '').replace('}', '')
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Special handling for translation text
    if is_translation:
        # Keep parentheses in translations as they might contain useful info
        # But remove if they're empty or just contain scribal notes
        text = re.sub(r'\(\s*\)', '', text)
        # Remove excessive punctuation in translations
        text = re.sub(r'[!?;:]+', '.', text)
    
    # Final cleanup
    text = text.replace('  ', ' ').strip()
    
    return text


def preprocess_dataset_entry(transliteration):
    """
    Preprocess a complete dataset entry (transliteration).
    
    Parameters:
    -----------
    transliteration : str
        Raw Akkadian transliteration text
    
    Returns:
    --------
    str : cleaned_transliteration
    """
    clean_translit = preprocess_akkadian_text(transliteration, is_translation=False)
    #clean_translation = preprocess_akkadian_text(translation, is_translation=True)
    
    return clean_translit # clean_translation


# Example usage with your data:
def preprocess_csv_data(df):
    """
    Preprocess a DataFrame containing 'transliteration' column.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with at least 'transliteration' column
    
    Returns:
    --------
    pandas.DataFrame : DataFrame with new 'clean_transliteration' column
    """
    import pandas as pd
    
    results = []
    for idx, row in df.iterrows():
        try:
            clean_translit = preprocess_dataset_entry(row['transliteration'])
            results.append({
                'clean_transliteration': clean_translit,
                'original_transliteration': row['transliteration']
            })
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            # Keep original if preprocessing fails
            results.append({
                'clean_transliteration': row['transliteration'],
                'original_transliteration': row['transliteration']
            })
    
    return pd.DataFrame(results)

In [5]:
data_pre_processed_pd = preprocess_csv_data(data_pd)
data_pre_processed_pd

,clean_transliteration,original_transliteration
0,um-ma ka3-ru-um ka3-ni-ia-ma a-na aa-qi2-il <b...,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,i-na mup-pi3-im aa a-limki ia-tu3 u„-mi3-im a-...,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,ki-ma mup-pi3-ni ta-a2a-me-a-ni a-ma-kam lu a-...,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,me-+e-er mup-pi3-ni a-na ka3-ar ka3-ar-ma u2 w...,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [6]:
MAX_LENGTH = 512
PREFIX = "translate Akkadian to English: "
BATCH_SIZE = 1

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['clean_transliteration'].astype(str).tolist()
        self.texts = [PREFIX + i for i in self.texts]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            #padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

test_dataset = InferenceDataset(data_pre_processed_pd, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- Inference Loop ---
print("Starting Inference...")
all_predictions = []

Starting Inference...


In [7]:
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
  
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])

  0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
# --- Submission ---
submission_Deep = pd.DataFrame({
    "id": data_pre_processed_pd.index,
    "translation": all_predictions
})

submission_Deep["translation"] = submission_Deep["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

submission_Deep.to_csv("submission.csv", index=False)
submission_Deep.head()

,id,translation
0,0,From the Kanesh colony to our messenger ... of...
1,1,As for the tablet in the City: Reckoned from t...
2,2,"In accordance with our letter, either he gave ..."
3,3,The sons of our letter I sent to every single ...
